# Notebook 10 — Learned Execution Selector

**Repo:** `int_serialization_benchmark-rml`  
**Layer:** `rml_extension/notebooks/`

Notebook 09 tested cross-hardware policy portability using transparent rule-based policies.

Notebook 10 introduces a lightweight learned execution selector:

- build features from RML metrics,
- predict selected policy,
- compare learned policy vs rule policy,
- estimate throughput / efficiency changes,
- inspect feature importance.

Constraint view:
> learned routing should improve policy selection without hiding the constraints that made the policy meaningful.

## Goals

1. Load Notebook 09 cross-hardware policy table.
2. Build a compact feature matrix.
3. Train simple interpretable models:
   - decision tree classifier
   - random forest classifier
4. Predict execution policy.
5. Compare:
   - rule-based selector
   - learned selector
   - best possible policy
6. Export CSV, JSON, Markdown report, and PNG figures.

This notebook is intentionally small and interpretable. It is a bridge toward learned routing, not a black-box production runtime.

In [ ]:
from pathlib import Path
import json
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

cwd = Path.cwd()
candidates = [
    cwd,
    cwd.parent,
    cwd.parent.parent,
    Path("/content/int_serialization_benchmark-rml"),
    Path("/content"),
]

REPO_ROOT = None
for c in candidates:
    if (c / "rml_extension").exists() or (c / "configs").exists():
        REPO_ROOT = c
        break

if REPO_ROOT is None:
    REPO_ROOT = cwd

RML_ROOT = REPO_ROOT / "rml_extension" if (REPO_ROOT / "rml_extension").exists() else REPO_ROOT

RESULTS_DIR = RML_ROOT / "results"
FIGURES_DIR = RML_ROOT / "figures"
REPORTS_DIR = RML_ROOT / "reports"

for d in [RESULTS_DIR, FIGURES_DIR, REPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("REPO_ROOT:", REPO_ROOT)
print("RML_ROOT:", RML_ROOT)

## Load Notebook 09 portability table

If missing, this notebook creates a fallback table with the same schema.

In [ ]:
portability_path = RESULTS_DIR / "notebook09_cross_hardware_policy_portability.csv"

if portability_path.exists():
    df = pd.read_csv(portability_path)
    print("Loaded:", portability_path)
else:
    print("Notebook 09 output not found; using fallback portability table.")
    hardware_profiles = ["scalar_reference", "avx2_linux", "avx512_linux", "neon_arm64", "cloud_vm_baseline"]
    regimes = ["low_entropy_repeating", "sequential_ids", "uniform_32bit", "zipfian_smallints", "clustered_ranges"]
    rows = []
    for hw in hardware_profiles:
        for i in range(6):
            for reg in regimes:
                pressure = {
                    "low_entropy_repeating": 0.02,
                    "sequential_ids": 0.30,
                    "uniform_32bit": 0.88,
                    "zipfian_smallints": 0.72,
                    "clustered_ranges": 0.98,
                }[reg]
                coherence = {
                    "low_entropy_repeating": 0.95,
                    "sequential_ids": 0.55,
                    "uniform_32bit": 0.08,
                    "zipfian_smallints": 0.42,
                    "clustered_ranges": 0.18,
                }[reg]
                if reg == "low_entropy_repeating":
                    pol = "coherent_local"
                elif reg == "uniform_32bit":
                    pol = "simd"
                elif reg == "clustered_ranges":
                    pol = "guarded_fallback"
                elif reg == "zipfian_smallints":
                    pol = "hybrid"
                else:
                    pol = "hybrid"
                if hw == "scalar_reference" and reg == "sequential_ids":
                    pol = "scalar"
                if hw == "avx512_linux" and reg in ["uniform_32bit", "zipfian_smallints"]:
                    pol = "simd"
                rows.append({
                    "hardware_profile": hw,
                    "architecture": "x86_64" if "avx" in hw or "scalar" in hw else ("arm64" if "neon" in hw else "virtualized"),
                    "window_id": len(rows),
                    "truth_regime": reg,
                    "coherence_score": coherence,
                    "hardware_pressure_proxy": pressure,
                    "selected_policy": pol,
                    "selected_throughput": {
                        "coherent_local": 1700,
                        "scalar": 1450,
                        "simd": 1900,
                        "guarded_fallback": 1200,
                        "hybrid": 1550,
                    }[pol] * (0.9 + 0.2 * np.random.default_rng(len(rows)).random()),
                    "best_possible_policy": pol,
                    "best_possible_throughput": 1950,
                    "selected_efficiency": 0.75 + 0.2 * np.random.default_rng(len(rows)+1).random(),
                })
    df = pd.DataFrame(rows)

df.head()

## Feature engineering

The learned selector uses transparent features:

- coherence score
- hardware pressure proxy
- architecture one-hot encoding
- regime one-hot encoding
- throughput/efficiency context if available

In [ ]:
work = df.copy()

for col in ["coherence_score", "hardware_pressure_proxy", "selected_throughput", "selected_efficiency"]:
    if col not in work.columns:
        work[col] = 0.0
    work[col] = pd.to_numeric(work[col], errors="coerce").fillna(0.0)

if "truth_regime" not in work.columns:
    work["truth_regime"] = "unknown"
if "architecture" not in work.columns:
    work["architecture"] = "unknown"
if "hardware_profile" not in work.columns:
    work["hardware_profile"] = "unknown"
if "selected_policy" not in work.columns:
    work["selected_policy"] = "hybrid"

feature_df = pd.get_dummies(
    work[[
        "coherence_score",
        "hardware_pressure_proxy",
        "selected_efficiency",
        "truth_regime",
        "architecture",
        "hardware_profile",
    ]],
    columns=["truth_regime", "architecture", "hardware_profile"],
    drop_first=False
)

target = work["selected_policy"].astype(str)

X = feature_df
y = target

print("Feature matrix:", X.shape)
print("Classes:", sorted(y.unique()))
X.head()

## Train/test split and models

Because this is a small synthetic/derived dataset, accuracy is only a sanity check.
Feature importance and mismatch locations are more useful than leaderboard metrics.

In [ ]:
stratify = y if y.value_counts().min() >= 2 else None

X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
    X, y, work.index,
    test_size=0.30,
    random_state=42,
    stratify=stratify
)

tree = DecisionTreeClassifier(max_depth=4, min_samples_leaf=2, random_state=42)
tree.fit(X_train, y_train)

forest = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
forest.fit(X_train, y_train)

y_pred_tree = tree.predict(X_test)
y_pred_forest = forest.predict(X_test)

tree_acc = accuracy_score(y_test, y_pred_tree)
forest_acc = accuracy_score(y_test, y_pred_forest)

print("Decision tree accuracy:", tree_acc)
print("Random forest accuracy:", forest_acc)

## Predict learned policy for all rows

In [ ]:
work["learned_policy_tree"] = tree.predict(X)
work["learned_policy_forest"] = forest.predict(X)

# Use forest as default learned selector.
work["learned_policy"] = work["learned_policy_forest"]
work["learned_matches_rule"] = work["learned_policy"] == work["selected_policy"]

# Estimate learned throughput by mapping policy to per-regime observed mean policy throughput.
policy_regime_mean = (
    work.groupby(["truth_regime", "selected_policy"])["selected_throughput"]
    .mean()
    .to_dict()
)

fallback_mean = float(work["selected_throughput"].mean())

def estimate_learned_tp(row):
    return policy_regime_mean.get((row["truth_regime"], row["learned_policy"]), fallback_mean)

work["learned_estimated_throughput"] = work.apply(estimate_learned_tp, axis=1)
work["learned_gain_vs_rule"] = work["learned_estimated_throughput"] - work["selected_throughput"]
work["learned_gain_pct_vs_rule"] = 100.0 * work["learned_gain_vs_rule"] / work["selected_throughput"].replace(0, np.nan)
work["learned_gain_pct_vs_rule"] = work["learned_gain_pct_vs_rule"].fillna(0.0)

work[["truth_regime", "hardware_profile", "selected_policy", "learned_policy", "learned_matches_rule", "learned_gain_pct_vs_rule"]].head()

## Export learned selector table

In [ ]:
csv_path = RESULTS_DIR / "notebook10_learned_execution_selector.csv"
json_path = RESULTS_DIR / "notebook10_learned_execution_selector.json"

work.to_csv(csv_path, index=False)
work.to_json(json_path, orient="records", indent=2)

print("Saved:", csv_path)
print("Saved:", json_path)

## Figure 1 — Learned vs rule-based policy agreement

In [ ]:
fig_path_1 = FIGURES_DIR / "notebook10_learned_policy_agreement.png"

agree = (
    work.groupby("hardware_profile", as_index=False)
    .agg(agreement_rate=("learned_matches_rule", "mean"))
    .sort_values("agreement_rate")
)

plt.figure(figsize=(9, 5))
plt.bar(agree["hardware_profile"], agree["agreement_rate"])
plt.xticks(rotation=45, ha="right")
plt.ylabel("Agreement rate")
plt.title("Learned Selector vs Rule Selector Agreement")
plt.tight_layout()
plt.savefig(fig_path_1, dpi=160)
plt.show()

print("Saved:", fig_path_1)

## Figure 2 — Feature importance

In [ ]:
fig_path_2 = FIGURES_DIR / "notebook10_feature_importance.png"

importances = pd.DataFrame({
    "feature": X.columns,
    "importance": forest.feature_importances_,
}).sort_values("importance", ascending=False).head(15)

plt.figure(figsize=(10, 6))
plt.bar(importances["feature"], importances["importance"])
plt.xticks(rotation=45, ha="right")
plt.ylabel("Importance")
plt.title("Learned Selector: Feature Importance")
plt.tight_layout()
plt.savefig(fig_path_2, dpi=160)
plt.show()

print("Saved:", fig_path_2)

## Figure 3 — Learned gain vs rule selector

In [ ]:
fig_path_3 = FIGURES_DIR / "notebook10_learned_gain_vs_rule.png"

gain = (
    work.groupby("truth_regime", as_index=False)
    .agg(mean_gain_pct=("learned_gain_pct_vs_rule", "mean"))
    .sort_values("mean_gain_pct")
)

plt.figure(figsize=(9, 5))
plt.bar(gain["truth_regime"], gain["mean_gain_pct"])
plt.xticks(rotation=45, ha="right")
plt.ylabel("Mean learned gain vs rule (%)")
plt.title("Learned Selector: Estimated Gain vs Rule Policy")
plt.tight_layout()
plt.savefig(fig_path_3, dpi=160)
plt.show()

print("Saved:", fig_path_3)

## Figure 4 — Policy confusion matrix

In [ ]:
fig_path_4 = FIGURES_DIR / "notebook10_policy_confusion_matrix.png"

labels = sorted(y.unique())
cm = confusion_matrix(work["selected_policy"], work["learned_policy"], labels=labels)

plt.figure(figsize=(7, 6))
plt.imshow(cm, aspect="auto")
plt.xticks(range(len(labels)), labels, rotation=45, ha="right")
plt.yticks(range(len(labels)), labels)
plt.xlabel("Learned policy")
plt.ylabel("Rule policy")
plt.title("Rule Policy vs Learned Policy Confusion Matrix")
plt.colorbar(label="Count")
plt.tight_layout()
plt.savefig(fig_path_4, dpi=160)
plt.show()

print("Saved:", fig_path_4)

## Figure 5 — Decision tree visualization

In [ ]:
fig_path_5 = FIGURES_DIR / "notebook10_decision_tree.png"

plt.figure(figsize=(18, 8))
plot_tree(
    tree,
    feature_names=list(X.columns),
    class_names=sorted(y.unique()),
    filled=False,
    rounded=True,
    fontsize=7
)
plt.title("Interpretable Learned Execution Selector")
plt.tight_layout()
plt.savefig(fig_path_5, dpi=180)
plt.show()

print("Saved:", fig_path_5)

## Lab-report summary

In [ ]:
report_path = REPORTS_DIR / "report_10_learned_execution_selector.md"

summary = {
    "rows": int(len(work)),
    "decision_tree_accuracy": float(tree_acc),
    "random_forest_accuracy": float(forest_acc),
    "learned_rule_agreement_rate": float(work["learned_matches_rule"].mean()),
    "mean_learned_gain_pct_vs_rule": float(work["learned_gain_pct_vs_rule"].mean()),
}

policy_counts = pd.crosstab(work["selected_policy"], work["learned_policy"])

lines = [
    "# Report 10 — Learned Execution Selector",
    "",
    "This report trains lightweight learned selectors for adaptive integer-serialization execution policy.",
    "",
    "Constraint view:",
    "> learned routing should improve policy selection without hiding the constraints that made the policy meaningful.",
    "",
    "## Generated outputs",
    "",
    f"- Metrics CSV: `{csv_path}`",
    f"- Metrics JSON: `{json_path}`",
    f"- Figure: `{fig_path_1}`",
    f"- Figure: `{fig_path_2}`",
    f"- Figure: `{fig_path_3}`",
    f"- Figure: `{fig_path_4}`",
    f"- Figure: `{fig_path_5}`",
    "",
    "## Summary",
    "",
    pd.DataFrame([summary]).to_markdown(index=False),
    "",
    "## Policy confusion table",
    "",
    policy_counts.to_markdown(),
    "",
    "## Top feature importances",
    "",
    importances.to_markdown(index=False),
    "",
    "## Interpretation",
    "",
    "- Learned selectors can recover policy structure from RML metrics and hardware-profile context.",
    "- Feature importance helps verify whether the model is using meaningful constraints.",
    "- Mismatches between learned and rule policies identify where hand-written heuristics may be brittle.",
    "- This notebook bridges transparent constraint rules and learned runtime routing.",
    "",
    "## Next step",
    "",
    "Notebook 11 can simulate online distribution classification: detect regime changes from windows before selecting execution paths.",
]

report_path.write_text("\n".join(lines))
print("Saved:", report_path)

## Optional: download output bundle in Colab

Uncomment the following cell if you are running this notebook in Google Colab and want to download generated outputs.

In [ ]:
# OPTIONAL COLAB DOWNLOAD
#
# EXPORT_NAME = "notebook10_learned_execution_selector_outputs.zip"
# export_path = RML_ROOT / EXPORT_NAME
#
# with zipfile.ZipFile(export_path, "w", zipfile.ZIP_DEFLATED) as zf:
#     for folder in [RESULTS_DIR, FIGURES_DIR, REPORTS_DIR]:
#         for p in folder.glob("notebook10_*"):
#             zf.write(p, arcname=str(p.relative_to(RML_ROOT)))
#         for p in folder.glob("report_10_*"):
#             zf.write(p, arcname=str(p.relative_to(RML_ROOT)))
#
# from google.colab import files
# files.download(str(export_path))